In [9]:
import os
import mlflow


mlflow.set_tracking_uri('file:./mlflow_experiments_store')


experiment_id = mlflow.get_experiment_by_name("Default").experiment_id


with mlflow.start_run(run_name='Default', experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metric("test_metric", 0)
    mlflow.log_artifact("test_artifact.txt", "test_artifact")

print(f"Run id запуска: {run_id}")

Run id запуска: 4e54164b768143df9c6acbe6cfa60439


In [12]:
import os
import mlflow
mlflow.set_tracking_uri('http://0.0.0.0:5000')

experiment_id = mlflow.get_experiment_by_name("Default").experiment_id

with mlflow.start_run(run_name="Default", experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    mlflow.log_metric("test_metric_sqlite", 0)
    mlflow.log_artifact("test_artifact.txt", "test_artifact_sqlite")


assert os.path.exists("mlflow_experiments_store_sqlite")
assert os.path.exists("mydb.sqlite")

In [10]:
!ls mlflow_experiments_store/0/c0f757c5f1ee4627b942f11dba1275a8/metrics

test_metric


In [11]:
!cat mlflow_experiments_store/0/c0f757c5f1ee4627b942f11dba1275a8/artifacts/test_artifact/test_artifact.txt

test_artifact

In [3]:
import os
import mlflow
import psycopg
import pandas as pd

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_SOURCE_HOST"), 
    "port": os.getenv("DB_SOURCE_PORT") ,
    "dbname": os.getenv("DB_SOURCE_NAME") ,
    "user": os.getenv("DB_SOURCE_USER"),
    "password": os.getenv("DB_SOURCE_PASSWORD") ,
}
assert all([var_value != "" for var_value in list(postgres_credentials.values())])

connection.update(postgres_credentials)

TABLE_NAME = "users_churn"


with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]


df = pd.DataFrame(data, columns=columns)

UndefinedTable: relation "users_churn" does not exist
LINE 1: SELECT * FROM users_churn
                      ^

In [ ]:
df.to_csv("users_churn.csv", index=False)

In [ ]:
EXPERIMENT_NAME = "churn_troston"
RUN_NAME = "data_check"

experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(experiment_id=experiment_id,
    run_name=RUN_NAME) as run:
    run_id = run.info.run_id
    
    
    mlflow.log_metrics(stats)
    

    mlflow.log_artifact('columns.txt', 'dataframe')
    mlflow.log_artifact('users_churn.csv', 'dataframe')


experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

run = mlflow.get_run(run_id)

assert run.info.status == "FINISHED"
